# 02 — Fetch Movie Metadata from OMDb API

This notebook queries the [OMDb API](https://www.omdbapi.com/) to retrieve metadata for each downloaded script: IMDb rating, Rotten Tomatoes score, genre, director, plot summary, etc.

**Run this notebook after** `01_scrape_scripts.ipynb`.

**Requires:** `OMDB_API_KEY` set in `.env` (free tier limit: 1,000 requests/day)

**Outputs:**
- `movie_metadata_temp.csv` — checkpoint file updated after each daily run
- `movie_metadata_clean.csv` — final clean file, created once all movies are processed

## 1. Setup — List Scripts and Define Helper Functions

Load environment variables, list all downloaded scripts, define `normalizar_titulo()` for filename-to-title conversion, and run a quick test query to verify the API key works.

In [ ]:
import os
#pip install python-dotenv
from dotenv import load_dotenv
load_dotenv()
import re
import requests
from thefuzz import process

# List all downloaded scripts
carpeta_completa = "scripts"
archivos = [f for f in os.listdir(carpeta_completa) if f.endswith('.txt')]

def normalizar_titulo(nombre_archivo):
    # Remove extension and replace underscores/dashes with spaces
    titulo = os.path.splitext(nombre_archivo)[0]
    titulo = titulo.replace('_', ' ').replace('-', ' ')
    # Remove special characters and extra whitespace
    titulo = re.sub(r'[^a-zA-Z0-9 ]', '', titulo)
    titulo = re.sub(r'\s+', ' ', titulo).strip()
    return titulo.lower()

# Normalize all filenames to readable titles
titulos_normalizados = [normalizar_titulo(f) for f in archivos]

print(f"Total de scripts descargados: {len(titulos_normalizados)}")
print("Ejemplo de títulos normalizados:")
for t in titulos_normalizados[:10]:
    print(f"- {t}")

# Define single-movie query function (used for quick tests below)
def obtener_rating_omdb(titulo, api_key):
    url = f"http://www.omdbapi.com/?t={titulo}&apikey={api_key}"
    res = requests.get(url)
    data = res.json()
    if data.get('Response') == 'True':
        return {
            'Title': data.get('Title'),
            'Year': data.get('Year'),
            'imdbRating': data.get('imdbRating'),
            'RottenTomatoes': next((r['Value'] for r in data.get('Ratings', []) if r['Source'] == 'Rotten Tomatoes'), None)
        }
    return None

api_key = os.getenv('OMDB_API_KEY')
if not api_key:
    raise ValueError("No se encontró la variable de entorno OMDB_API_KEY. Defínela antes de ejecutar el código.")

# Quick sanity check — query first 5 titles
resultados = []
for t in titulos_normalizados[:5]:
    print(f"\nConsultando: {t}")
    info = obtener_rating_omdb(t, api_key)
    print(info)
    resultados.append({'titulo_normalizado': t, 'info_omdb': info})

## 2. Debug — Test API Connection

Isolate a specific title to confirm the request URL format and API key authorization are correct before running the bulk query.

In [3]:
# CELDA DE DEBUG - Investigar "Black Snake Moan"
import time
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("="*70)
print("DEBUG: Buscando 'Black Snake Moan'")
print("="*70)

# 1. Buscar el archivo
archivo_buscado = [f for f in archivos if 'black' in f.lower() and 'snake' in f.lower()]
print(f"\nArchivos encontrados con 'black snake': {archivo_buscado}")

# 2. Normalizar
if archivo_buscado:
    titulo_normalizado = normalizar_titulo(archivo_buscado[0])
    print(f"Título normalizado: '{titulo_normalizado}'")
else:
    titulo_normalizado = "black snake moan"
    print(f"Usando título manual: '{titulo_normalizado}'")

# 3. Intentar consulta directa
print(f"\nIntentando consulta a OMDb...")
print(f"URL: http://www.omdbapi.com/?t={titulo_normalizado}&apikey=***")

try:
    url = f"http://www.omdbapi.com/?t={titulo_normalizado}&apikey={api_key}"
    res = requests.get(url, timeout=5)
    res.raise_for_status()
    data = res.json()
    
    print(f"\nStatus Code: {res.status_code}")
    print(f"Response: {data.get('Response')}")
    print(f"Title: {data.get('Title')}")
    print(f"Year: {data.get('Year')}")
    print(f"Error (si existe): {data.get('Error')}")
    
    if data.get('Response') == 'True':
        print("\n✅ ENCONTRADA")
        print(f"   Título: {data.get('Title')}")
        print(f"   Año: {data.get('Year')}")
        print(f"   IMDb Rating: {data.get('imdbRating')}")
        print(f"   Rotten Tomatoes: {next((r['Value'] for r in data.get('Ratings', []) if r['Source'] == 'Rotten Tomatoes'), 'N/A')}")
    else:
        print(f"\n❌ NO ENCONTRADA: {data.get('Error')}")
        
except Exception as e:
    print(f"\n❌ ERROR en la consulta: {type(e).__name__}")
    print(f"   Detalles: {str(e)}")

print("\n" + "="*70)

DEBUG: Buscando 'Black Snake Moan'

Archivos encontrados con 'black snake': ['black_snake_moan.txt']
Título normalizado: 'black snake moan'

Intentando consulta a OMDb...
URL: http://www.omdbapi.com/?t=black snake moan&apikey=***

❌ ERROR en la consulta: HTTPError
   Detalles: 401 Client Error: Unauthorized for url: http://www.omdbapi.com/?t=black%20snake%20moan&apikey=70f95252



## 3. Bulk Query with Daily Checkpoint

Queries OMDb for all scripts, respecting the 1,000 requests/day free tier limit. Progress is saved to `movie_metadata_temp.csv` after each run so the process can be resumed the next day.

**Run this cell once per day** until all movies are processed.

In [5]:
# CELDA 3: Consultar TODOS los scripts con CHECKPOINT
import pandas as pd
import time
from datetime import datetime
import json
import os

print("="*70)
print("CONSULTANDO TODOS LOS SCRIPTS A OMDB (Con reinicio automático)")
print("="*70)
# ─────────────────────────────────────────────────────────────────────
# 1. VERIFICAR Y CARGAR CHECKPOINT (para reanudar progreso)
# ─────────────────────────────────────────────────────────────────────
CHECKPOINT_FILE = "omdb_checkpoint.json"
RESULTADO_CSV = "movie_metadata_temp.csv"
MAX_REQUESTS_POR_DIA = 1000

# Función para cargar checkpoint
def cargar_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {
        'fecha_ultimo': None,
        'consultas_hoy': 0,
        'indice_ultimo': -1,
        'indices_consultados': []
    }

# Función para guardar checkpoint
def guardar_checkpoint(checkpoint):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f, indent=2)

# Cargar checkpoint
checkpoint = cargar_checkpoint()
fecha_hoy = datetime.now().strftime("%Y-%m-%d")

# Resetear contador si es un día diferente
if checkpoint['fecha_ultimo'] != fecha_hoy:
    print(f"\n🔄 Nuevo día detectado: {fecha_hoy}")
    checkpoint['fecha_ultimo'] = fecha_hoy
    checkpoint['consultas_hoy'] = 0
    print(f"   Contador de requests reiniciado: 0/{MAX_REQUESTS_POR_DIA}")
else:
    print(f"\n Continuando desde: {fecha_hoy}")
    print(f"   Requests usados hoy: {checkpoint['consultas_hoy']}/{MAX_REQUESTS_POR_DIA}")
    print(f"   Requests disponibles: {MAX_REQUESTS_POR_DIA - checkpoint['consultas_hoy']}")

# ─────────────────────────────────────────────────────────────────────
# 2. CARGAR RESULTADOS PREVIOS (si existen)
# ─────────────────────────────────────────────────────────────────────

if os.path.exists(RESULTADO_CSV):
    df_resultados_prev = pd.read_csv(RESULTADO_CSV)
    print(f"\n✅ Resultados previos encontrados: {len(df_resultados_prev)} películas ya consultadas")
    indices_ya_consultados = set(df_resultados_prev.index)
else:
    df_resultados_prev = None
    indices_ya_consultados = set()
    print(f"\n📝 Iniciando consultas desde cero")

# ─────────────────────────────────────────────────────────────────────
# 3. FUNCIÓN PARA CONSULTAR OMDB
# ─────────────────────────────────────────────────────────────────────

def obtener_rating_omdb_robusto(titulo, api_key, timeout=3):
    try:
        url = f"http://www.omdbapi.com/?t={titulo}&apikey={api_key}"
        res = requests.get(url, timeout=timeout)
        res.raise_for_status()
        data = res.json()
        
        if data.get('Response') == 'True':
            return {
                'Title': data.get('Title'),
                'Year': data.get('Year'),
                'imdbRating': data.get('imdbRating'),
                'RottenTomatoes': next((r['Value'] for r in data.get('Ratings', []) if r['Source'] == 'Rotten Tomatoes'), 'N/A'),
                'Plot': data.get('Plot'),
                'Director': data.get('Director'),
                'Actors': data.get('Actors'),
                'Genre': data.get('Genre'),
                'Tipo': data.get('Type'),
                'Response': 'Found'
            }
        else:
            return {
                'Title': None,
                'Year': None,
                'imdbRating': None,
                'RottenTomatoes': None,
                'Plot': None,
                'Director': None,
                'Actors': None,
                'Genre': None,
                'Tipo': None,
                'Response': data.get('Error', 'Not Found')
            }
    except Exception as e:
        return {
            'Title': None,
            'Year': None,
            'imdbRating': None,
            'RottenTomatoes': None,
            'Plot': None,
            'Director': None,
            'Actors': None,
            'Genre': None,
            'Tipo': None,
            'Response': f'Error: {str(e)}'
        }

# ─────────────────────────────────────────────────────────────────────
# 4. IDENTIFICAR PELÍCULAS PENDIENTES
# ─────────────────────────────────────────────────────────────────────

total_scripts = len(titulos_normalizados)
películas_pendientes = [i for i in range(total_scripts) if i not in indices_ya_consultados]

print(f"\n📊 Estado actual:")
print(f"   Total películas: {total_scripts}")
print(f"   Ya consultadas: {len(indices_ya_consultados)}")
print(f"   Pendientes: {len(películas_pendientes)}")
print(f"   Disponibles hoy: {MAX_REQUESTS_POR_DIA - checkpoint['consultas_hoy']}")

# ─────────────────────────────────────────────────────────────────────
# 5. CONSULTAR PELÍCULAS PENDIENTES (respetando límite diario)
# ─────────────────────────────────────────────────────────────────────

resultados_nuevos = []
requests_disponibles = MAX_REQUESTS_POR_DIA - checkpoint['consultas_hoy']

if requests_disponibles <= 0:
    print(f"\n⚠️  LÍMITE DIARIO ALCANZADO (1000 requests)")
    print(f"   Vuelve mañana para continuar con {len(películas_pendientes)} películas restantes")
else:
    # Limitar consultas a las disponibles hoy
    películas_a_consultar = películas_pendientes[:requests_disponibles]
    
    print(f"\n Consultando {len(películas_a_consultar)} películas hoy...")
    print(f"   (De {len(películas_pendientes)} pendientes)")
    
    inicio = datetime.now()
    
    for idx, idx_película in enumerate(películas_a_consultar):
        titulo_norm = titulos_normalizados[idx_película]
        archivo_original = archivos[idx_película]
        
        # Mostrar progreso
        if (idx + 1) % 50 == 0:
            tiempo_transcurrido = (datetime.now() - inicio).total_seconds() / 60
            print(f"   Progreso: {idx + 1}/{len(películas_a_consultar)} - {tiempo_transcurrido:.1f} min")
        
        # Consultar OMDb
        info = obtener_rating_omdb_robusto(titulo_norm, api_key)
        
        # Ruta del script
        ruta_script = os.path.join(carpeta_completa, archivo_original)
        
        # Guardar resultado
        resultados_nuevos.append({
            'indice': idx_película,
            'script_filename': archivo_original,
            'script_path': ruta_script,
            'titulo_normalizado': titulo_norm,
            'omdb_title': info['Title'],
            'omdb_year': info['Year'],
            'omdb_rating_imdb': info['imdbRating'],
            'omdb_rating_rottenomatoes': info['RottenTomatoes'],
            'omdb_plot': info['Plot'],
            'omdb_director': info['Director'],
            'omdb_actors': info['Actors'],
            'omdb_genre': info['Genre'],
            'omdb_type': info['Tipo'],
            'omdb_response': info['Response'],
            'timestamp': datetime.now().isoformat()
        })
        
        # Pequeño delay
        time.sleep(0.1)
    
    tiempo_total = (datetime.now() - inicio).total_seconds() / 60
    
    # ─────────────────────────────────────────────────────────────────────
    # 6. GUARDAR RESULTADOS NUEVOS Y ACTUALIZAR CHECKPOINT
    # ─────────────────────────────────────────────────────────────────────
    
    df_nuevos = pd.DataFrame(resultados_nuevos)
    
    # Combinar con resultados previos
    if df_resultados_prev is not None:
        df_omdb = pd.concat([df_resultados_prev, df_nuevos], ignore_index=True)
    else:
        df_omdb = df_nuevos
    
    # Guardar en archivo temporal
    df_omdb.to_csv(RESULTADO_CSV, index=False)
    
    # Actualizar checkpoint
    checkpoint['consultas_hoy'] += len(resultados_nuevos)
    checkpoint['indice_ultimo'] = películas_a_consultar[-1] if películas_a_consultar else checkpoint['indice_ultimo']
    guardar_checkpoint(checkpoint)
    
    print(f"\n✅ Consultas completadas en {tiempo_total:.1f} minutos")
    print(f"   Nuevos registros: {len(resultados_nuevos)}")
    print(f"   Total acumulado: {len(df_omdb)}")
    print(f"\n📊 Uso de requests hoy:")
    print(f"   Consumidos: {checkpoint['consultas_hoy']}/{MAX_REQUESTS_POR_DIA}")
    print(f"   Disponibles mañana: {MAX_REQUESTS_POR_DIA}")
    
    # ─────────────────────────────────────────────────────────────────────
    # 7. MOSTRAR PREVIEW
    # ─────────────────────────────────────────────────────────────────────
    
    print("\n" + "="*70)
    print("PREVIEW DEL DATAFRAME ACUMULADO")
    print("="*70)
    print(df_omdb.head())
    
    print("\n" + "="*70)
    print("RESUMEN ACUMULADO")
    print("="*70)
    print(f"Total registros: {len(df_omdb)}")
    print(f"Películas encontradas: {(df_omdb['omdb_response'] == 'Found').sum()}")
    print(f"No encontradas: {(df_omdb['omdb_response'] != 'Found').sum()}")
    
    if len(películas_pendientes) > len(películas_a_consultar):
        print(f"\n⏳ Quedan {len(películas_pendientes) - len(películas_a_consultar)} películas por consultar")
        print(f"   Ejecuta esta celda mañana para continuar ✅")


CONSULTANDO TODOS LOS SCRIPTS A OMDB (Con reinicio automático)

🔄 Nuevo día detectado: 2026-01-25
   Contador de requests reiniciado: 0/1000

📝 Iniciando consultas desde cero

📊 Estado actual:
   Total películas: 918
   Ya consultadas: 0
   Pendientes: 918
   Disponibles hoy: 1000

 Consultando 918 películas hoy...
   (De 918 pendientes)
   Progreso: 50/918 - 0.1 min
   Progreso: 100/918 - 0.3 min
   Progreso: 150/918 - 0.4 min
   Progreso: 200/918 - 0.5 min
   Progreso: 250/918 - 0.7 min
   Progreso: 300/918 - 0.8 min
   Progreso: 350/918 - 0.9 min
   Progreso: 400/918 - 1.1 min
   Progreso: 450/918 - 1.2 min
   Progreso: 500/918 - 1.3 min
   Progreso: 550/918 - 1.5 min
   Progreso: 600/918 - 1.6 min
   Progreso: 650/918 - 1.7 min
   Progreso: 700/918 - 1.9 min
   Progreso: 750/918 - 2.0 min
   Progreso: 800/918 - 2.1 min
   Progreso: 850/918 - 2.3 min
   Progreso: 900/918 - 2.4 min

✅ Consultas completadas en 2.4 minutos
   Nuevos registros: 918
   Total acumulado: 918

📊 Uso de requ

## 4. Clean and Save Results

Load the checkpoint CSV, convert data types (IMDb ratings to `float`, Rotten Tomatoes percentages to `float` in [0, 1]), and save a clean version once all movies have been queried.

In [6]:
# CELDA 4: Guardar dataframe final y crear estructura optimizada para ML

# Verificar si existe el archivo temporal
if os.path.exists("movie_metadata_temp.csv"):
    df_omdb = pd.read_csv("movie_metadata_temp.csv")
    print(f"✅ Dataframe cargado desde checkpoint: {len(df_omdb)} registros")
else:
    print("⚠️  No se encontró archivo temporal. Asegúrate de ejecutar la Celda 3 primero.")
    df_omdb = None

if df_omdb is not None:
    # A. Guardar CSV TEMPORAL (mantiene el checkpoint)
    print(f"\n📝 Archivo temporal (checkpoint): movie_metadata_temp.csv")
    
    # B. Crear versión limpia (solo películas encontradas)
    df_encontradas = df_omdb[df_omdb['omdb_response'] == 'Found'].copy()
    print(f"\n✅ Películas encontradas: {len(df_encontradas)} de {len(df_omdb)}")
    
    # Convertir ratings a numéricas
    df_encontradas['omdb_rating_imdb'] = pd.to_numeric(df_encontradas['omdb_rating_imdb'], errors='coerce')
    
    # Rotten Tomatoes cleanup
    df_encontradas['omdb_rating_rottenomatoes'] = df_encontradas['omdb_rating_rottenomatoes'].apply(
        lambda x: float(x.rstrip('%')) / 100 if isinstance(x, str) and x != 'N/A' else None
    )
    df_encontradas['omdb_year'] = pd.to_numeric(df_encontradas['omdb_year'], errors='coerce')
    
    # C. Guardar versión limpia SOLO cuando todas estén consultadas
    if len(df_omdb) == 1298:  # Si completó todas las películas
        csv_limpio = "movie_metadata_clean.csv"
        df_encontradas.to_csv(csv_limpio, index=False)
        print(f"✅ Metadata limpia (FINAL) guardada en: {csv_limpio}")
    else:
        print(f"⏳ Consultadas: {len(df_omdb)}/1298. Metadata final se guardará cuando terminen todas.")
    
    # D. Mostrar estadísticas
    print("\n" + "="*70)
    print("ESTADÍSTICAS DE RATINGS (Películas encontradas)")
    print("="*70)
    
    imdb_ratings = df_encontradas['omdb_rating_imdb'].dropna()
    rt_ratings = df_encontradas['omdb_rating_rottenomatoes'].dropna()
    
    if len(imdb_ratings) > 0:
        print(f"IMDb Rating:")
        print(f"  Promedio: {imdb_ratings.mean():.2f}")
        print(f"  Min: {imdb_ratings.min():.2f}, Max: {imdb_ratings.max():.2f}")
        print(f"  Películas con rating: {len(imdb_ratings)}/{len(df_encontradas)}")
    
    if len(rt_ratings) > 0:
        print(f"\nRotten Tomatoes Rating:")
        print(f"  Promedio: {rt_ratings.mean():.2%}")
        print(f"  Películas con rating: {len(rt_ratings)}/{len(df_encontradas)}")
    
    print("\n" + "="*70)
    print("RECOMENDACIÓN PARA MACHINE LEARNING")
    print("="*70)
    print("""
✅ ESTRUCTURA RECOMENDADA (Escalable y eficiente)
────────────────────────────────────────────────────
Archivos creados:

1. movie_metadata_temp.csv (CHECKPOINT - en desarrollo)
   - Se actualiza cada día con nuevos resultados
   - NO usar para ML aún

2. movie_metadata_clean.csv (FINAL - cuando terminen todas)
   - Se crea cuando consultes todas las 1298 películas
   - Contiene: titulo, año, ratings, género, directores
   - Usa esta para ML

3. todos_los_scripts/ (Carpeta con scripts)
   - 1298 archivos .txt con contenido de películas
   - Referenciados por 'script_path' en CSV

Ventajas:
  ✓ Separar metadata de scripts (memory-efficient)
  ✓ Cargar datos en batches durante entrenamiento
  ✓ Escalable a millones de películas
  ✓ Ideal para deep learning (LSTM, Transformers, etc.)

Próximos pasos:
  1. Ejecuta Celda 3 cada día hasta completar 1298
  2. Cuando llegues a 1298, tendrás movie_metadata_clean.csv
  3. Usa ese CSV + scripts para entrenar modelo
    """)


✅ Dataframe cargado desde checkpoint: 918 registros

📝 Archivo temporal (checkpoint): movie_metadata_temp.csv

✅ Películas encontradas: 898 de 918
⏳ Consultadas: 918/1298. Metadata final se guardará cuando terminen todas.

ESTADÍSTICAS DE RATINGS (Películas encontradas)
IMDb Rating:
  Promedio: 6.91
  Min: 2.40, Max: 9.00
  Películas con rating: 883/898

Rotten Tomatoes Rating:
  Promedio: 68.91%
  Películas con rating: 809/898

RECOMENDACIÓN PARA MACHINE LEARNING

✅ ESTRUCTURA RECOMENDADA (Escalable y eficiente)
────────────────────────────────────────────────────
Archivos creados:

1. movie_metadata_temp.csv (CHECKPOINT - en desarrollo)
   - Se actualiza cada día con nuevos resultados
   - NO usar para ML aún

2. movie_metadata_clean.csv (FINAL - cuando terminen todas)
   - Se crea cuando consultes todas las 1298 películas
   - Contiene: titulo, año, ratings, género, directores
   - Usa esta para ML

3. todos_los_scripts/ (Carpeta con scripts)
   - 1298 archivos .txt con contenido de

## 5. Dataset Overview and `MovieDataset` Class

Show rating statistics, display a sample of the collected data, and define `MovieDataset` — a helper class for loading scripts alongside their metadata in batches (useful for model training).

In [8]:
# CELDA 5: Cargar y mostrar datos de OMDb (movie_metadata)
import pandas as pd

# Priorizar archivo limpio si existe, sino usar el temporal
if os.path.exists("movie_metadata_clean.csv"):
    df_metadata = pd.read_csv("movie_metadata_clean.csv")
    archivo_usado = "movie_metadata_clean.csv"
elif os.path.exists("movie_metadata_temp.csv"):
    df_metadata = pd.read_csv("movie_metadata_temp.csv")
    archivo_usado = "movie_metadata_temp.csv"
else:
    print("⚠️  No se encontró ningún archivo de metadata. Ejecuta Celda 3 primero.")
    df_metadata = None

if df_metadata is not None:
    print("="*70)
    print(f"DATASET DE OMDB: {archivo_usado}")
    print("="*70)
    print(f"\n✅ Dataset cargado: {len(df_metadata)} registros")
    print(f"📊 Columnas disponibles: {list(df_metadata.columns)}")
    
    print("\n" + "="*70)
    print("SAMPLE DEL DATAFRAME - Primeras 10 filas")
    print("="*70)
    print(df_metadata.head(10))
    
    print("\n" + "="*70)
    print("INFORMACIÓN DETALLADA")
    print("="*70)
    print(df_metadata.info())
    
    print("\n" + "="*70)
    print("ESTADÍSTICAS DE RATINGS")
    print("="*70)
    # Convertir ratings a numéricas si no están ya
    if 'omdb_rating_imdb' in df_metadata.columns:
        df_metadata['omdb_rating_imdb_numeric'] = pd.to_numeric(df_metadata['omdb_rating_imdb'], errors='coerce')
        ratings_imdb = df_metadata['omdb_rating_imdb_numeric'].dropna()
        if len(ratings_imdb) > 0:
            print(f"\nIMDb Rating:")
            print(f"  Promedio: {ratings_imdb.mean():.2f}")
            print(f"  Desviación estándar: {ratings_imdb.std():.2f}")
            print(f"  Min: {ratings_imdb.min():.2f}, Max: {ratings_imdb.max():.2f}")
            print(f"  Películas con rating: {len(ratings_imdb)}/{len(df_metadata)}")
    
    if 'omdb_rating_rottenomatoes' in df_metadata.columns:
        print(f"\nRotten Tomatoes Rating:")
        print(f"  Películas con rating: {df_metadata['omdb_rating_rottenomatoes'].notna().sum()}/{len(df_metadata)}")
    
    print("\n" + "="*70)
    print("ANÁLISIS DE RESPUESTAS (Películas encontradas vs no encontradas)")
    print("="*70)
    if 'omdb_response' in df_metadata.columns:
        print(df_metadata['omdb_response'].value_counts())
    
    print("\n" + "="*70)
    print("MUESTRA DE PELÍCULAS ENCONTRADAS")
    print("="*70)
    if 'omdb_response' in df_metadata.columns:
        df_encontradas = df_metadata[df_metadata['omdb_response'] == 'Found']
        print(f"Total encontradas: {len(df_encontradas)}")
        print("\nPrimeras 5 películas encontradas:")
        cols_mostrar = ['script_filename', 'omdb_title', 'omdb_year', 'omdb_rating_imdb', 'omdb_genre']
        cols_existentes = [col for col in cols_mostrar if col in df_encontradas.columns]
        print(df_encontradas[cols_existentes].head(5))

# ─────────────────────────────────────────────────────────────────────
# Clase para cargar películas completas (metadata + scripts)
# ─────────────────────────────────────────────────────────────────────

class MovieDataset:
    """Cargador eficiente para datos de películas + scripts"""
    
    def __init__(self, csv_path="movie_metadata_clean.csv"):
        self.df = pd.read_csv(csv_path)
        print(f"\n✅ Metadata cargado: {len(self.df)} películas")
    
    def load_script(self, idx):
        """Cargar script por índice"""
        row = self.df.iloc[idx]
        try:
            with open(row['script_path'], 'r', encoding='utf-8') as f:
                return f.read()
        except:
            return None
    
    def get_batch(self, indices, load_scripts=True):
        """Obtener batch de películas con o sin scripts"""
        batch = self.df.iloc[indices].copy()
        
        if load_scripts:
            batch['script_text'] = [self.load_script(i) for i in indices]
        
        return batch
    
    def get_by_rating_range(self, min_rating=5, max_rating=10):
        """Filtrar por rating de IMDb"""
        mask = (self.df['omdb_rating_imdb'] >= min_rating) & \
               (self.df['omdb_rating_imdb'] <= max_rating)
        return self.df[mask]

print("\n" + "="*70)
print("PRÓXIMOS PASOS")
print("="*70)
print("""
✅ Ahora que tienes metadata de OMDb, puedes:

1. Continuar ejecutando Celda 3 cada día hasta completar todas las 1298 películas

2. Una vez completo, hacer merge con el dataset de Oscar:
   dataset = MovieDataset("movie_metadata_clean.csv")
   df_oscar = pd.read_csv("full_data.csv", delimiter='\\t')
   df_merged = df_oscar.merge(dataset.df, 
                              left_on='Film',
                              right_on='omdb_title',
                              how='left')

3. Usar df_merged para entrenar tu modelo de ML:
   - Features: Contenido de scripts (script_path)
   - Target: omdb_rating_imdb
""")


DATASET DE OMDB: movie_metadata_temp.csv

✅ Dataset cargado: 918 registros
📊 Columnas disponibles: ['indice', 'script_filename', 'script_path', 'titulo_normalizado', 'omdb_title', 'omdb_year', 'omdb_rating_imdb', 'omdb_rating_rottenomatoes', 'omdb_plot', 'omdb_director', 'omdb_actors', 'omdb_genre', 'omdb_type', 'omdb_response', 'timestamp']

SAMPLE DEL DATAFRAME - Primeras 10 filas
   indice                          script_filename  \
0       0                     black_snake_moan.txt   
1       1  mr_blandings_builds_his_dream_house.txt   
2       2            tinker_tailor_soldier_spy.txt   
3       3      wall_street:_money_never_sleeps.txt   
4       4           evil_dead_ii:_dead_by_dawn.txt   
5       5                             repo_man.txt   
6       6                          pitch_black.txt   
7       7                   dances_with_wolves.txt   
8       8       war_for_the_planet_of_the_apes.txt   
9       9                          hudson_hawk.txt   

                   